# GraphIQ — AI-Powered Knowledge Graph Reasoning Engine
### Advanced Machine Learning Project Notebook

---

## Workflow Overview

```
Raw Documents
     ↓
Text Chunking
     ↓
Named Entity Recognition (spaCy)
     ↓
Relation Extraction
     ↓
Knowledge Graph (Neo4j)
     ↓
Graph Embeddings (sentence-transformers)
     ↓
Vector Index (FAISS)
     ↓
AI Reasoning (Groq LLM)
     ↓
Natural Language Answers
```

## Table of Contents
1. [Setup & Imports](#1-setup)
2. [Text Chunking](#2-chunking)
3. [Named Entity Recognition](#3-ner)
4. [Relation Extraction](#4-re)
5. [Knowledge Graph — Neo4j](#5-graph)
6. [Graph Analysis & Visualization](#6-analysis)
7. [Node Embeddings](#7-embeddings)
8. [Semantic Search — FAISS](#8-faiss)
9. [AI Query Engine](#9-query)
10. [Results Summary](#10-results)

---
## 1. Setup & Imports <a id='1-setup'></a>

Load all required libraries and connect to services.

In [ ]:
import os, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from pathlib import Path
from dotenv import load_dotenv

import spacy
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv('../api/.env')

print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print('All imports successful!')

In [ ]:
# Load spaCy transformer model for NER
nlp = spacy.load('en_core_web_trf')
print(f'spaCy model : {nlp.meta["name"]} v{nlp.meta["version"]}')
print(f'Pipeline    : {nlp.pipe_names}')

In [ ]:
# Connect to Neo4j
driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD', 'password'))
)

def run_query(cypher, params={}):
    with driver.session() as s:
        return [r.data() for r in s.run(cypher, params)]

result = run_query("RETURN 'Connected!' AS msg")
print(f'Neo4j : {result[0]["msg"]}')

---
## 2. Text Chunking <a id='2-chunking'></a>

Large documents are split into overlapping chunks before NLP processing.
Overlapping ensures entities that appear near chunk boundaries are not missed.

In [ ]:
def chunk_text(text, chunk_size=400, overlap=50):
    """
    Split text into fixed-size chunks with overlap.
    - chunk_size : max characters per chunk
    - overlap    : characters shared between consecutive chunks
    """
    text = re.sub(r'\s+', ' ', text).strip()
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size - overlap
    return chunks


# Load sample document
sample_path = Path('../data/graphiq_synopsis.txt')
if sample_path.exists():
    text = sample_path.read_text(encoding='utf-8')
else:
    text = (
        "GraphIQ is an AI-Powered Knowledge Graph Reasoning Engine. "
        "It uses spaCy and BERT for Named Entity Recognition. "
        "Neo4j is used as the graph database. "
        "Node2Vec generates graph embeddings. "
        "FAISS enables fast vector similarity search. "
        "FastAPI provides the REST API backend. "
        "React and Vite power the frontend. "
        "Groq and Llama3 serve as the AI reasoning engine. "
        "GraphIQ supports risk analysis and compliance monitoring. "
        "The system demonstrates explainable multi-hop reasoning."
    )

chunks = chunk_text(text)
print(f'Document length : {len(text)} characters')
print(f'Total chunks    : {len(chunks)}')
print(f'Chunk size      : 400 chars  |  Overlap: 50 chars')
print()
for i, c in enumerate(chunks):
    print(f'Chunk {i+1} ({len(c)} chars) : {c[:80]}...')

---
## 3. Named Entity Recognition <a id='3-ner'></a>

spaCy's transformer model scans each chunk and labels named entities.
Each entity gets a type: PERSON, ORG, GPE, DATE, etc.

In [ ]:
ENTITY_TYPE_MAP = {
    'PERSON'     : 'person',
    'ORG'        : 'organization',
    'GPE'        : 'location',
    'DATE'       : 'date',
    'MONEY'      : 'financial',
    'EVENT'      : 'event',
    'PRODUCT'    : 'product',
    'WORK_OF_ART': 'document',
}

def extract_entities(text):
    doc = nlp(text)
    seen, entities = set(), []
    for ent in doc.ents:
        name = ent.text.strip()
        if name in seen or len(name) < 2:
            continue
        seen.add(name)
        entities.append({
            'name' : name,
            'type' : ENTITY_TYPE_MAP.get(ent.label_, 'other'),
            'label': ent.label_
        })
    return entities


# Run NER on all chunks
all_entities = []
for i, chunk in enumerate(chunks):
    ents = extract_entities(chunk)
    all_entities.extend(ents)
    print(f'Chunk {i+1} : {len(ents)} entities')

# Deduplicate by name
unique_entities = list({e['name']: e for e in all_entities}.values())
df_entities = pd.DataFrame(unique_entities)

print(f'\nTotal unique entities : {len(unique_entities)}')
print()
print(df_entities.to_string(index=False))

In [ ]:
# Plot entity type distribution
TYPE_COLORS = {
    'person'      : '#a78bfa',
    'organization': '#00f5d4',
    'location'    : '#34d399',
    'date'        : '#fbbf24',
    'event'       : '#f87171',
    'financial'   : '#60a5fa',
    'document'    : '#fb923c',
    'other'       : '#94a3b8',
}

counts = df_entities['type'].value_counts()
bar_colors = [TYPE_COLORS.get(t, '#94a3b8') for t in counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0a1628')
for ax in (ax1, ax2):
    ax.set_facecolor('#0a1628')
    ax.spines[:].set_color('#1e3a5f')
    ax.tick_params(colors='#9ab8cc')

# Bar
bars = ax1.bar(counts.index, counts.values, color=bar_colors, width=0.6)
ax1.set_title('Entity Type Count', color='#00f5d4', fontsize=12)
ax1.set_ylabel('Count', color='#9ab8cc')
for bar, v in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.05,
             str(v), ha='center', color='#c8daea', fontsize=9)

# Pie
wedges, texts, autos = ax2.pie(
    counts.values, labels=counts.index, colors=bar_colors,
    autopct='%1.0f%%', startangle=90,
    textprops={'color':'#9ab8cc', 'fontsize':9}
)
for a in autos:
    a.set_color('#020818'); a.set_fontsize(8)
ax2.set_title('Entity Type Share', color='#00f5d4', fontsize=12)

fig.suptitle('NER Results — Entity Distribution', color='#e2e8f0', fontsize=14)
plt.tight_layout()
plt.savefig('../data/entity_distribution.png', dpi=150,
            bbox_inches='tight', facecolor='#0a1628')
plt.show()

---
## 4. Relation Extraction <a id='4-re'></a>

For each sentence, if two or more entities appear together,
the main verb is extracted as the relationship type.
This creates (head, relation, tail) triples — the building blocks of the knowledge graph.

In [ ]:
def extract_relations(text, entities):
    """
    Heuristic relation extraction:
    - For each sentence, find entity pairs that co-occur
    - Extract the ROOT verb as the relation label
    Returns list of (head, relation, tail) triples.
    """
    doc = nlp(text)
    relations = []
    for sent in doc.sents:
        found = [e for e in entities if e['name'] in sent.text]
        if len(found) < 2:
            continue
        verb = 'related_to'
        for token in sent:
            if token.pos_ == 'VERB' and token.dep_ in ('ROOT', 'relcl', 'advcl'):
                verb = token.lemma_.lower()[:25]
                break
        head, tail = found[0], found[1]
        if head['name'] != tail['name']:
            relations.append({
                'head'    : head['name'],
                'relation': verb,
                'tail'    : tail['name'],
            })
    return relations


# Extract relations across all chunks
all_relations = []
for chunk in chunks:
    ents = extract_entities(chunk)
    rels = extract_relations(chunk, ents)
    all_relations.extend(rels)

df_relations = pd.DataFrame(all_relations)
print(f'Total relations extracted : {len(df_relations)}')
if not df_relations.empty:
    print()
    print(df_relations.to_string(index=False))

---
## 5. Knowledge Graph — Neo4j <a id='5-graph'></a>

Entities become nodes. Relations become directed edges.
Neo4j stores everything using the Cypher query language.
`MERGE` is used instead of `CREATE` to avoid duplicate nodes.

In [ ]:
# Query current graph statistics
node_count  = run_query('MATCH (n) RETURN count(n) AS cnt')[0]['cnt']
edge_count  = run_query('MATCH ()-[r]->() RETURN count(r) AS cnt')[0]['cnt']
node_types  = run_query('MATCH (n:Entity) RETURN DISTINCT n.type AS type, count(n) AS cnt ORDER BY cnt DESC')
edge_types  = run_query('MATCH ()-[r]->() RETURN DISTINCT type(r) AS rel, count(r) AS cnt ORDER BY cnt DESC')
sources     = run_query('MATCH (n:Entity) RETURN DISTINCT n.source AS src, count(n) AS cnt')

print('=== Neo4j Graph Statistics ===')
print(f'Nodes : {node_count}')
print(f'Edges : {edge_count}')

print('\nNode types:')
for r in node_types:
    print(f'  {r["type"]:<20} {r["cnt"]}')

print('\nEdge (relation) types:')
for r in edge_types:
    print(f'  {r["rel"]:<20} {r["cnt"]}')

print('\nIngested sources:')
for s in sources:
    print(f'  {s["src"]} — {s["cnt"]} nodes')

In [ ]:
# Example Cypher queries
print('=== Multi-hop Cypher Query ===')
result = run_query(
    'MATCH (a:Entity)-[r]->(b:Entity) '
    'RETURN a.name AS from, type(r) AS relation, b.name AS to '
    'LIMIT 10'
)
df_q = pd.DataFrame(result)
if not df_q.empty:
    print(df_q.to_string(index=False))
else:
    print('No edges found — ingest a document first.')

---
## 6. Graph Analysis & Visualization <a id='6-analysis'></a>

NetworkX is used to analyze graph structure — centrality, density, and connectivity.
matplotlib visualizes the graph with nodes colored by entity type.

In [ ]:
# Build NetworkX graph from Neo4j
nodes_data = run_query('MATCH (n:Entity) RETURN n.name AS name, n.type AS type')
edges_data = run_query('MATCH (a:Entity)-[r]->(b:Entity) RETURN a.name AS src, type(r) AS rel, b.name AS dst')

G = nx.DiGraph()
for n in nodes_data:
    G.add_node(n['name'], type=n['type'])
for e in edges_data:
    G.add_edge(e['src'], e['dst'], label=e['rel'])

print(f'Nodes    : {G.number_of_nodes()}')
print(f'Edges    : {G.number_of_edges()}')
print(f'Density  : {nx.density(G):.4f}')
if G.number_of_nodes() > 0:
    avg_deg = sum(dict(G.degree()).values()) / G.number_of_nodes()
    print(f'Avg degree : {avg_deg:.2f}')

In [ ]:
# Visualize knowledge graph
if G.number_of_nodes() > 0:
    fig, ax = plt.subplots(figsize=(14, 9))
    fig.patch.set_facecolor('#0a1628')
    ax.set_facecolor('#0a1628')

    pos = nx.spring_layout(G, k=2.5, seed=42)
    node_colors = [
        TYPE_COLORS.get(G.nodes[n].get('type', 'other'), '#94a3b8')
        for n in G.nodes()
    ]

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=700, alpha=0.9)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#1e3a5f',
                           arrows=True, arrowsize=15, width=1.5,
                           connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7,
                            font_color='#e2e8f0', font_weight='bold')

    legend_patches = [
        mpatches.Patch(color=c, label=t.title())
        for t, c in TYPE_COLORS.items()
        if any(G.nodes[n].get('type') == t for n in G.nodes())
    ]
    ax.legend(handles=legend_patches, loc='upper left',
              facecolor='#03091c', labelcolor='#9ab8cc', edgecolor='#1e3a5f')
    ax.set_title('Knowledge Graph Visualization', color='#00f5d4', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../data/knowledge_graph.png', dpi=150,
                bbox_inches='tight', facecolor='#0a1628')
    plt.show()
else:
    print('Graph is empty — ingest a document first.')

In [ ]:
# Centrality analysis
# Degree centrality = how many connections a node has
# Betweenness centrality = how often a node acts as a bridge
if G.number_of_nodes() > 0:
    degree_c     = nx.degree_centrality(G)
    betweenness  = nx.betweenness_centrality(G)
    in_deg       = dict(G.in_degree())
    out_deg      = dict(G.out_degree())

    df_centrality = pd.DataFrame({
        'node'             : list(degree_c.keys()),
        'degree_centrality': list(degree_c.values()),
        'betweenness'      : [betweenness[n] for n in degree_c],
        'in_degree'        : [in_deg[n]  for n in degree_c],
        'out_degree'       : [out_deg[n] for n in degree_c],
    }).sort_values('degree_centrality', ascending=False)

    print('=== Top 10 Nodes by Degree Centrality ===')
    print(df_centrality.head(10).to_string(index=False))

    # Plot
    top = df_centrality.head(10)
    fig, ax = plt.subplots(figsize=(11, 5))
    fig.patch.set_facecolor('#0a1628')
    ax.set_facecolor('#0a1628')
    ax.barh(top['node'], top['degree_centrality'], color='#00f5d4', alpha=0.8)
    ax.set_title('Top 10 Nodes — Degree Centrality', color='#00f5d4', fontsize=12)
    ax.set_xlabel('Degree Centrality', color='#9ab8cc')
    ax.tick_params(colors='#9ab8cc')
    ax.spines[:].set_color('#1e3a5f')
    plt.tight_layout()
    plt.savefig('../data/centrality.png', dpi=150,
                bbox_inches='tight', facecolor='#0a1628')
    plt.show()

---
## 7. Node Embeddings <a id='7-embeddings'></a>

Each node (entity) is converted into a dense vector using sentence-transformers.
The vector captures the semantic meaning of the entity's name and type.
PCA reduces the high-dimensional vectors to 2D for visualization.

In [ ]:
# Load embedding model
print('Loading sentence transformer...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded — output dimensions: 384')

In [ ]:
# Generate embeddings for every node
if G.number_of_nodes() > 0:
    node_list  = list(G.nodes())
    node_texts = [
        f"{n} is a {G.nodes[n].get('type', 'entity')}"
        for n in node_list
    ]

    print(f'Generating embeddings for {len(node_list)} nodes...')
    node_embeddings = embed_model.encode(node_texts, show_progress_bar=True)

    print(f'Embedding matrix shape : {node_embeddings.shape}')
    print(f'  Rows    = {node_embeddings.shape[0]} nodes')
    print(f'  Columns = {node_embeddings.shape[1]} dimensions per node')
else:
    print('Graph is empty.')

In [ ]:
# PCA — reduce 384-dim vectors to 2D for visualization
if G.number_of_nodes() > 1:
    pca     = PCA(n_components=2)
    reduced = pca.fit_transform(node_embeddings)

    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor('#0a1628')
    ax.set_facecolor('#0a1628')

    pt_colors = [
        TYPE_COLORS.get(G.nodes[n].get('type', 'other'), '#94a3b8')
        for n in node_list
    ]
    ax.scatter(reduced[:, 0], reduced[:, 1],
               c=pt_colors, s=180, alpha=0.85, zorder=3)
    for i, name in enumerate(node_list):
        ax.annotate(name, (reduced[i, 0], reduced[i, 1]),
                    xytext=(6, 4), textcoords='offset points',
                    fontsize=7, color='#c8daea')

    legend_patches = [
        mpatches.Patch(color=c, label=t.title())
        for t, c in TYPE_COLORS.items()
        if any(G.nodes[n].get('type') == t for n in node_list)
    ]
    ax.legend(handles=legend_patches, facecolor='#03091c',
              labelcolor='#9ab8cc', edgecolor='#1e3a5f')
    ax.set_title('Node Embeddings — PCA (2D Projection)', color='#00f5d4', fontsize=13)
    ax.set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]:.1%} variance)', color='#9ab8cc')
    ax.set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]:.1%} variance)', color='#9ab8cc')
    ax.tick_params(colors='#9ab8cc')
    ax.spines[:].set_color('#1e3a5f')
    ax.grid(color='#1e3a5f', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../data/embeddings_pca.png', dpi=150,
                bbox_inches='tight', facecolor='#0a1628')
    plt.show()
    print(f'Total variance explained : {sum(pca.explained_variance_ratio_):.1%}')

In [ ]:
# Cosine similarity heatmap between all node embeddings
if G.number_of_nodes() > 1:
    sim_matrix  = cosine_similarity(node_embeddings)
    short_names = [n[:14] for n in node_list]

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('#0a1628')
    ax.set_facecolor('#0a1628')

    im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='Cosine Similarity')
    ax.set_xticks(range(len(short_names)))
    ax.set_yticks(range(len(short_names)))
    ax.set_xticklabels(short_names, rotation=45, ha='right',
                       fontsize=7, color='#9ab8cc')
    ax.set_yticklabels(short_names, fontsize=7, color='#9ab8cc')
    ax.set_title('Node Embedding Similarity Matrix', color='#00f5d4', fontsize=13)
    plt.tight_layout()
    plt.savefig('../data/similarity_matrix.png', dpi=150,
                bbox_inches='tight', facecolor='#0a1628')
    plt.show()

---
## 8. Semantic Search — FAISS <a id='8-faiss'></a>

FAISS (Facebook AI Similarity Search) builds a fast index over all node embeddings.
Given a natural language query, it returns the most semantically similar nodes
without requiring exact keyword matches.

In [ ]:
# Build FAISS index
if G.number_of_nodes() > 0:
    dim   = node_embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)

    # Normalize before indexing (converts L2 to cosine similarity)
    norms      = np.linalg.norm(node_embeddings, axis=1, keepdims=True)
    normalized = (node_embeddings / norms).astype('float32')
    index.add(normalized)

    print(f'FAISS index built')
    print(f'  Type       : {type(index).__name__}')
    print(f'  Vectors    : {index.ntotal}')
    print(f'  Dimensions : {dim}')

In [ ]:
# Semantic search function
def semantic_search(query, top_k=5):
    """
    Encode query → search FAISS index → return top-k similar nodes.
    """
    q_vec = embed_model.encode([query]).astype('float32')
    q_vec = q_vec / np.linalg.norm(q_vec)
    distances, indices = index.search(q_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < len(node_list):
            results.append({
                'node'      : node_list[idx],
                'type'      : G.nodes[node_list[idx]].get('type', 'other'),
                'similarity': round(float(1 - dist / 2), 4)
            })
    return results


# Run test queries
if G.number_of_nodes() > 0:
    test_queries = [
        'What database stores the graph?',
        'Which library handles vector search?',
        'What framework is used for the API?'
    ]
    for q in test_queries:
        print(f'Query : "{q}"')
        for r in semantic_search(q, top_k=3):
            print(f'  → {r["node"]:<25} [{r["type"]}]  sim={r["similarity"]}')
        print()

---
## 9. AI Query Engine <a id='9-query'></a>

The full reasoning pipeline:
1. Fetch all entities and relationships from Neo4j
2. Format them as a structured context string
3. Send context + user query to Groq (Llama3)
4. LLM reasons over the graph and returns an explainable answer with a reasoning path

In [ ]:
# Initialize Groq client
GROQ_KEY = os.getenv('GROQ_API_KEY')
if GROQ_KEY:
    groq_client = Groq(api_key=GROQ_KEY)
    print('Groq client ready — model: llama-3.3-70b-versatile')
else:
    print('GROQ_API_KEY not found in .env')

In [ ]:
def graphiq_query(user_query):
    """
    Full pipeline:
      Neo4j graph  →  context string  →  LLM prompt  →  reasoned answer
    """
    # Step 1 — fetch graph
    nodes = run_query('MATCH (n:Entity) RETURN n.name AS name, n.type AS type')
    edges = run_query(
        'MATCH (a:Entity)-[r]->(b:Entity) '
        'RETURN a.name AS head, type(r) AS relation, b.name AS tail'
    )

    # Step 2 — build context
    nodes_str = ', '.join(f"{n['name']}({n['type']})" for n in nodes)
    edges_str = '; '.join(f"{e['head']}-[{e['relation']}]->{e['tail']}" for e in edges)

    prompt = f"""You are a knowledge graph reasoning engine.

GRAPH:
ENTITIES: {nodes_str}
RELATIONSHIPS: {edges_str}

QUERY: "{user_query}"

Respond with:
**ANSWER**: [direct answer using graph data]
**PATH**: [Entity → [relation] → Entity]
**CONFIDENCE**: [High / Medium / Low]
**INSIGHT**: [one extra pattern found in the graph]"""

    # Step 3 — LLM reasoning
    response = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=600
    )
    answer = response.choices[0].message.content

    print(f'Query: {user_query}')
    print('─' * 60)
    print(answer)
    print()
    return answer


# Run example queries
if GROQ_KEY and G.number_of_nodes() > 0:
    queries = [
        'What technologies are used in this system?',
        'What is the purpose of this system?',
    ]
    for q in queries:
        graphiq_query(q)
        print('=' * 60)

---
## 10. Results Summary <a id='10-results'></a>

Final dashboard summarizing all pipeline outputs.

In [ ]:
# 4-panel results dashboard
COLORS = ['#00f5d4','#a78bfa','#fbbf24','#34d399','#f87171','#60a5fa','#fb923c']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0a1628')
fig.suptitle('GraphIQ — Results Dashboard', color='#00f5d4', fontsize=16, y=0.98)

for ax in axes.flat:
    ax.set_facecolor('#0a1628')
    ax.spines[:].set_color('#1e3a5f')
    ax.tick_params(colors='#9ab8cc')

# Panel 1 — Entity type distribution
if not df_entities.empty:
    c = df_entities['type'].value_counts()
    axes[0,0].bar(c.index, c.values,
                  color=[TYPE_COLORS.get(t,'#94a3b8') for t in c.index], width=0.6)
    axes[0,0].set_title('Entities by Type', color='#00f5d4', fontsize=11)
    axes[0,0].set_ylabel('Count', color='#9ab8cc')

# Panel 2 — Pipeline stages
stages     = ['Ingestion','NER','Relation RE','Graph DB','Embeddings','FAISS','LLM']
completion = [100, 100, 100, 100, 100, 100, 100]
bar_colors = ['#34d399' if c == 100 else '#fbbf24' for c in completion]
axes[0,1].barh(stages, completion, color=bar_colors)
axes[0,1].set_xlim(0, 115)
axes[0,1].set_title('Pipeline Completion', color='#00f5d4', fontsize=11)
for i, v in enumerate(completion):
    axes[0,1].text(v+1, i, f'{v}%', va='center', color='#c8daea', fontsize=8)

# Panel 3 — Graph metrics bar
metrics = ['Nodes','Edges','Node Types','Rel Types']
values  = [
    G.number_of_nodes(),
    G.number_of_edges(),
    len(set(nx.get_node_attributes(G,'type').values())),
    len(set(nx.get_edge_attributes(G,'label').values()))
]
axes[1,0].bar(metrics, values, color=COLORS[:4], width=0.5)
axes[1,0].set_title('Knowledge Graph Metrics', color='#00f5d4', fontsize=11)
for i, v in enumerate(values):
    axes[1,0].text(i, v+0.1, str(v), ha='center', color='#c8daea', fontsize=10)

# Panel 4 — Tech stack
tech  = ['spaCy','Neo4j','FAISS','FastAPI','React','Groq LLM']
roles = ['NER + RE','Graph DB','Vector Search','REST API','Frontend UI','Reasoning']
axes[1,1].axis('off')
axes[1,1].set_title('Technology Stack', color='#00f5d4', fontsize=11, pad=10)
for i, (t, r) in enumerate(zip(tech, roles)):
    y = 0.88 - i * 0.14
    col = COLORS[i % len(COLORS)]
    axes[1,1].text(0.04, y, f'● {t}', transform=axes[1,1].transAxes,
                   fontsize=11, color=col, fontweight='bold')
    axes[1,1].text(0.42, y, r, transform=axes[1,1].transAxes,
                   fontsize=10, color='#9ab8cc')

plt.tight_layout()
plt.savefig('../data/results_summary.png', dpi=150,
            bbox_inches='tight', facecolor='#0a1628')
plt.show()
print('Results dashboard saved to data/results_summary.png')

In [ ]:
# Final stats printout
print('=== FINAL PIPELINE SUMMARY ===')
print(f'Chunks processed    : {len(chunks)}')
print(f'Entities extracted  : {len(unique_entities)}')
print(f'Relations extracted : {len(all_relations)}')
print(f'Graph nodes         : {G.number_of_nodes()}')
print(f'Graph edges         : {G.number_of_edges()}')
print(f'Graph density       : {nx.density(G):.4f}')
if G.number_of_nodes() > 0:
    print(f'Embedding dims      : {node_embeddings.shape[1]}')
    print(f'FAISS vectors       : {index.ntotal}')
print()
print('Saved charts:')
for f in ['entity_distribution','knowledge_graph','centrality',
          'embeddings_pca','similarity_matrix','results_summary']:
    path = Path(f'../data/{f}.png')
    status = 'saved' if path.exists() else 'not found'
    print(f'  data/{f}.png — {status}')

driver.close()
print('\nNeo4j connection closed.')